# Prompt Templates

In [1]:
import pandas as pd
import duckdb as db
from openai import OpenAI
import os

In [12]:
user_template = "Write a SQL query that returns: {}"

In [13]:
system_template = """
Given the following SQL table, your job is to write queries based on a user's request. \n
CREATE TABLE {} ({}) \n
"""

Load the csv file into a Pandas DataFrame

In [14]:
nbi_2025_ak = pd.read_csv("../data/nbi/AK25.csv")

In [15]:
nbi_2025_ak['STRUCTURE_NUMBER_008'] = nbi_2025_ak['STRUCTURE_NUMBER_008'].astype(str)
nbi_2025_ak['COUNTY_CODE_003'] = nbi_2025_ak['COUNTY_CODE_003'].astype(str)

In [24]:
question = "What are the total number of bridges by county? Return the results ordered by the county code."

tbl_name = "nbi_2025_ak"

In [25]:
tbl_description = db.sql("DESCRIBE SELECT * FROM " + tbl_name + ";")
tbl_description

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ STATE_CODE_001         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ STRUCTURE_NUMBER_008   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ RECORD_TYPE_005A       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ ROUTE_PREFIX_005B      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ SERVICE_LEVEL_005C     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ ROUTE_NUMBER_005D      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ DIRECTION_005E         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ HIGHWAY_DISTRICT_002   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ COUNTY_CODE_003        │ V

In [26]:
col_attr = tbl_description.df()[["column_name", "column_type"]]
col_attr["column_joint"] = col_attr["column_name"] + " " + col_attr["column_type"]
tbl_schema = (
    str(list(col_attr["column_joint"].values))
    .replace("[", "")
    .replace("]", "")
    .replace("'", "")
)

print(tbl_schema)

STATE_CODE_001 BIGINT, STRUCTURE_NUMBER_008 VARCHAR, RECORD_TYPE_005A BIGINT, ROUTE_PREFIX_005B BIGINT, SERVICE_LEVEL_005C BIGINT, ROUTE_NUMBER_005D VARCHAR, DIRECTION_005E BIGINT, HIGHWAY_DISTRICT_002 VARCHAR, COUNTY_CODE_003 VARCHAR, PLACE_CODE_004 BIGINT, FEATURES_DESC_006A VARCHAR, CRITICAL_FACILITY_006B DOUBLE, FACILITY_CARRIED_007 VARCHAR, LOCATION_009 VARCHAR, MIN_VERT_CLR_010 DOUBLE, KILOPOINT_011 DOUBLE, BASE_HWY_NETWORK_012 DOUBLE, LRS_INV_ROUTE_013A DOUBLE, SUBROUTE_NO_013B DOUBLE, LAT_016 BIGINT, LONG_017 BIGINT, DETOUR_KILOS_019 BIGINT, TOLL_020 BIGINT, MAINTENANCE_021 BIGINT, OWNER_022 BIGINT, FUNCTIONAL_CLASS_026 BIGINT, YEAR_BUILT_027 BIGINT, TRAFFIC_LANES_ON_028A BIGINT, TRAFFIC_LANES_UND_028B BIGINT, ADT_029 BIGINT, YEAR_ADT_030 BIGINT, DESIGN_LOAD_031 VARCHAR, APPR_WIDTH_MT_032 DOUBLE, MEDIAN_CODE_033 BIGINT, DEGREES_SKEW_034 BIGINT, STRUCTURE_FLARED_035 BIGINT, RAILINGS_036A VARCHAR, TRANSITIONS_036B VARCHAR, APPR_RAIL_036C VARCHAR, APPR_RAIL_END_036D VARCHAR, HISTO

In [27]:
user = user_template.format(question)
print(user)

Write a SQL query that returns: What are the total number of bridges by county? Return the results ordered by the county code.


In [28]:
system = system_template.format(tbl_name,tbl_schema)
print(system)


Given the following SQL table, your job is to write queries based on a user's request. 

CREATE TABLE nbi_2025_ak (STATE_CODE_001 BIGINT, STRUCTURE_NUMBER_008 VARCHAR, RECORD_TYPE_005A BIGINT, ROUTE_PREFIX_005B BIGINT, SERVICE_LEVEL_005C BIGINT, ROUTE_NUMBER_005D VARCHAR, DIRECTION_005E BIGINT, HIGHWAY_DISTRICT_002 VARCHAR, COUNTY_CODE_003 VARCHAR, PLACE_CODE_004 BIGINT, FEATURES_DESC_006A VARCHAR, CRITICAL_FACILITY_006B DOUBLE, FACILITY_CARRIED_007 VARCHAR, LOCATION_009 VARCHAR, MIN_VERT_CLR_010 DOUBLE, KILOPOINT_011 DOUBLE, BASE_HWY_NETWORK_012 DOUBLE, LRS_INV_ROUTE_013A DOUBLE, SUBROUTE_NO_013B DOUBLE, LAT_016 BIGINT, LONG_017 BIGINT, DETOUR_KILOS_019 BIGINT, TOLL_020 BIGINT, MAINTENANCE_021 BIGINT, OWNER_022 BIGINT, FUNCTIONAL_CLASS_026 BIGINT, YEAR_BUILT_027 BIGINT, TRAFFIC_LANES_ON_028A BIGINT, TRAFFIC_LANES_UND_028B BIGINT, ADT_029 BIGINT, YEAR_ADT_030 BIGINT, DESIGN_LOAD_031 VARCHAR, APPR_WIDTH_MT_032 DOUBLE, MEDIAN_CODE_033 BIGINT, DEGREES_SKEW_034 BIGINT, STRUCTURE_FLARED_03

In [29]:
openai_api_key = os.environ.get("OPENAI_API_KEY")
max_tokens = 5000
client = OpenAI(api_key=openai_api_key)

In [30]:
response = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "system", "content": system}, 
              {"role": "user", "content": user}],
    max_completion_tokens=max_tokens,
)

print(response.choices[0].message.content)

SELECT
  COUNTY_CODE_003 AS county_code,
  COUNT(*) AS total_bridges
FROM nbi_2025_ak
WHERE RECORD_TYPE_005A = 1
GROUP BY COUNTY_CODE_003
ORDER BY COUNTY_CODE_003;


In [31]:
query = response.choices[0].message.content
db.sql(query)

┌─────────────┬───────────────┐
│ county_code │ total_bridges │
│   varchar   │     int64     │
├─────────────┼───────────────┤
│ 100.0       │            20 │
│ 105.0       │           103 │
│ 110.0       │            47 │
│ 122.0       │           102 │
│ 13.0        │            16 │
│ 130.0       │           107 │
│ 150.0       │            25 │
│ 158.0       │             1 │
│ 16.0        │             8 │
│ 164.0       │            10 │
│   ·         │             · │
│   ·         │             · │
│   ·         │             · │
│ 282.0       │            16 │
│ 290.0       │            94 │
│ 50.0        │            13 │
│ 60.0        │             2 │
│ 63.0        │            65 │
│ 66.0        │            53 │
│ 68.0        │            27 │
│ 70.0        │             3 │
│ 90.0        │           112 │
│ nan         │             1 │
├─────────────┴───────────────┤
│     31 rows (20 shown)      │
└─────────────────────────────┘

In [33]:
question2 = "What are the total number of bridges by county? Return the results ordered by the number of bridges."

user2 = user_template.format(question2)

In [34]:
response2 = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "system", "content": system}, {"role": "user", "content": user2}],
    max_completion_tokens=max_tokens,
)

print(response2.choices[0].message.content)

SELECT
  COUNTY_CODE_003 AS county_code,
  COUNT(*) AS total_bridges
FROM nbi_2025_ak
GROUP BY COUNTY_CODE_003
ORDER BY total_bridges DESC, county_code;


In [35]:
query2 = response2.choices[0].message.content
db.sql(query2)

┌─────────────┬───────────────┐
│ county_code │ total_bridges │
│   varchar   │     int64     │
├─────────────┼───────────────┤
│ 198.0       │           226 │
│ 195.0       │           137 │
│ 20.0        │           134 │
│ 170.0       │           121 │
│ 90.0        │           112 │
│ 130.0       │           107 │
│ 105.0       │           103 │
│ 122.0       │           102 │
│ 290.0       │            94 │
│ 63.0        │            65 │
│  ·          │             · │
│  ·          │             · │
│  ·          │             · │
│ 282.0       │            16 │
│ 50.0        │            13 │
│ 164.0       │            10 │
│ 16.0        │             8 │
│ 188.0       │             7 │
│ 230.0       │             6 │
│ 70.0        │             3 │
│ 60.0        │             2 │
│ 158.0       │             1 │
│ nan         │             1 │
├─────────────┴───────────────┤
│     31 rows (20 shown)      │
└─────────────────────────────┘